In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [104]:
df = pd.read_csv('Steam.csv')

In [105]:
# Convertir fechas
df['release_date'] = pd.to_datetime(df['release_date'])

# Variables temporales y cuatrimestre
df['year'] = df['release_date'].dt.year
df['month'] = df['release_date'].dt.month
df['quarter'] = df['release_date'].dt.quarter
df['periodo'] = df['year'].astype(str) + '-Q' + df['quarter'].astype(str)

# Segmentación modelo de negocio
df['type'] = df['price'].apply(lambda x: 'Free to Play' if x == 0 else 'Premium')

# variable owners
def clean_owners(owner_str):
    try:
        low, high = owner_str.split('-')
        return (int(low) + int(high)) / 2
    except:
        return 0

df['owners_numeric'] = df['owners'].apply(clean_owners)

# Segmentación escala de producción

publisher_counts = df['publisher'].value_counts()

# Diccionario de publicadores AAA/AA para evitar falsos Indies
aaa_keywords = [
    'electronic arts', 'ubisoft', 'bethesda', 'square enix', 'sega', 
    'capcom', 'xbox', 'playstation', 'rockstar', '2k', 'activision', 
    'bandai namco', 'warner bros', 'sony', 'microsoft', 'cd projekt'
]

def classify_scale(publisher):
    pub_str = str(publisher).lower()
    
    # Detección por nombre de publicador grande
    if any(keyword in pub_str for keyword in aaa_keywords):
        return 'AAA/AA'
        
    # Detección por volumen de catálogo (>= 10)
    if publisher_counts.get(publisher, 0) >= 10:
        return 'AAA/AA'
        
    return 'Indie'

df['production_scale'] = df['publisher'].apply(classify_scale)
df['production_scale'] = pd.Categorical(df['production_scale'], categories=['Indie', 'AAA/AA'], ordered=True)

# Plataformas
df['platform_count'] = df['platforms'].apply(lambda x: len(str(x).split(';')))

# Positividad
df['total_ratings'] = df['positive_ratings'] + df['negative_ratings']
df['positivity_ratio'] = df['positive_ratings'] / df['total_ratings']
df['positivity_ratio'] = df['positivity_ratio'].fillna(0)

# Costo por hora
df['median_playtime_hours'] = df['median_playtime'] / 60
df['cost_per_hour'] = np.where(
    df['median_playtime_hours'] > 0, 
    df['price'] / df['median_playtime_hours'], 
    0
)

# Géneros y categorías
df['genres_list'] = df['genres'].apply(lambda x: str(x).split(';'))
df['categories_list'] = df['categories'].apply(lambda x: str(x).split(';'))

# Variable multijugador
df['is_multiplayer'] = df['categories'].apply(lambda x: 1 if 'Multi-player' in str(x) else 0)

# Variable binaria idioma
df['english_support'] = df['english'].apply(lambda x: 'Sí' if x == 1 else 'No')

df.to_csv('steam_cleaned_full.csv', index=False)

In [106]:
df = pd.read_csv('steam_cleaned_full.csv')

df['production_scale'] = pd.Categorical(df['production_scale'], categories=['Indie', 'AAA/AA'], ordered=True)

In [107]:
# Gráfico 1: Violín estático
fig1 = px.violin(df, x='production_scale', y='positivity_ratio', color='production_scale',
                 box=True, points=False, template="plotly_dark",
                 color_discrete_map={'Indie': '#00CC96', 'AAA/AA': '#EF553B'},
                 title="Distribución de Positividad: Indie vs AAA/AA",
                 labels={'production_scale': 'Escala de Producción', 'positivity_ratio': 'Ratio de Positividad'})
fig1.update_layout(title_font_size=20, margin=dict(t=60, b=40, l=40, r=40))

print("""
GRÁFICO 1: Distribución de Positividad (Violín Estático)
- Qué es: Gráfico de densidad probabilística de la métrica de satisfacción.
- Qué se ve: Concentración de calidad percibida por escala de producción corregida.
- Elementos: Eje X (Escala), Eje Y (Positividad).
""")

fig1.show()



GRÁFICO 1: Distribución de Positividad (Violín Estático)
- Qué es: Gráfico de densidad probabilística de la métrica de satisfacción.
- Qué se ve: Concentración de calidad percibida por escala de producción corregida.
- Elementos: Eje X (Escala), Eje Y (Positividad).



In [108]:
# Gráfico 2: Barras
users_scale = df.groupby('production_scale', as_index=False)['owners_numeric'].sum()
fig2 = px.bar(users_scale, x='production_scale', y='owners_numeric', color='production_scale',
              template="plotly_dark", text_auto='.2s',
              color_discrete_map={'Indie': '#00CC96', 'AAA/AA': '#EF553B'},
              title="Usuarios Totales Acumulados: Indie vs AAA/AA",
              labels={'production_scale': 'Escala de Producción', 'owners_numeric': 'Usuarios Totales'})
fig2.update_traces(textposition='outside', textfont_size=14)
fig2.update_layout(title_font_size=20, showlegend=False)

print("""
GRÁFICO 2: Usuarios Totales Acumulados (Barras)
- Qué es: Volumen de mercado estático por escala.
- Qué se ve: Distribución total de adopción.
- Elementos: Eje X (Escala), Eje Y (Millones de usuarios).
""")

fig2.show()


GRÁFICO 2: Usuarios Totales Acumulados (Barras)
- Qué es: Volumen de mercado estático por escala.
- Qué se ve: Distribución total de adopción.
- Elementos: Eje X (Escala), Eje Y (Millones de usuarios).



In [109]:
fig3 = px.pie(df, names='type', values='owners_numeric', hole=0.6,
              color='type', template="plotly_dark", 
              color_discrete_sequence=['#00F0FF', '#FF003C'],
              title="Cuota de Mercado: F2P vs Premium (Total Usuarios)")

fig3.update_traces(hoverinfo='label+percent', textinfo='label+percent', textfont_size=16,
                   marker=dict(line=dict(color='#111111', width=2)))
fig3.update_layout(title_font_size=20, annotations=[dict(text='Mercado', x=0.5, y=0.5, font_size=20, showarrow=False)])

print("""
GRÁFICO 3: Cuota de Mercado F2P vs Premium (Dona)
- Qué es: Proporción de mercado estática.
- Qué se ve: Distribución del volumen total de jugadores según modelo de monetización.
- Elementos: Sectores (F2P/Premium), Porcentajes.
""")

fig3.show()


GRÁFICO 3: Cuota de Mercado F2P vs Premium (Dona)
- Qué es: Proporción de mercado estática.
- Qué se ve: Distribución del volumen total de jugadores según modelo de monetización.
- Elementos: Sectores (F2P/Premium), Porcentajes.



In [110]:
df_premium = df[df['type'] == 'Premium'].copy()

fig4 = px.scatter(df_premium, x='price', y='median_playtime_hours', 
                  size='owners_numeric', color='production_scale', hover_name='name',
                  opacity=0.6, template="plotly_dark", size_max=40,
                  marginal_x="histogram", marginal_y="histogram",
                  color_discrete_map={'Indie': '#00CC96', 'AAA/AA': '#EF553B'},
                  title="Precio vs Tiempo de Juego: Concentración de Rentabilidad (Premium)",
                  labels={'price': 'Precio ($)', 'median_playtime_hours': 'Tiempo Mediano (Horas)', 'production_scale': 'Escala'},
                  range_x=[0, 60], range_y=[0, 100])

fig4.update_layout(title_font_size=20)

print("""
GRÁFICO 4: Precio vs Tiempo de Juego (Dispersión con Histogramas Marginales)
- Qué es: Mapa estático de dispersión con distribución lateral y superior.
- Qué se ve: Correlación entre precio y horas jugadas con la nueva segmentación.
- Elementos: Eje X (Precio), Eje Y (Horas), Tamaño (Usuarios), Color (Escala AAA/AA vs Indie).
""")

fig4.show()


GRÁFICO 4: Precio vs Tiempo de Juego (Dispersión con Histogramas Marginales)
- Qué es: Mapa estático de dispersión con distribución lateral y superior.
- Qué se ve: Correlación entre precio y horas jugadas con la nueva segmentación.
- Elementos: Eje X (Precio), Eje Y (Horas), Tamaño (Usuarios), Color (Escala AAA/AA vs Indie).



In [111]:
# Filtro y extracción
df_played = df[df['median_playtime_hours'] > 0].copy()
df_played['primary_genre'] = df_played['genres'].apply(lambda x: str(x).replace(',', ';').split(';')[0].strip())

all_genres = df_played.groupby('primary_genre', as_index=False).agg(
    median_playtime=('median_playtime_hours', 'median')
).sort_values('median_playtime', ascending=False)

fig5 = px.bar(all_genres, x='primary_genre', y='median_playtime', color='median_playtime',
              color_continuous_scale='Inferno', template="plotly_dark", text_auto='.1f',
              title="Retención de Jugadores: Todos los Géneros (Excluyendo inactivos)",
              labels={'primary_genre': 'Género Principal', 'median_playtime': 'Horas Medianas'})

fig5.update_traces(textposition='outside')
fig5.update_layout(title_font_size=20, xaxis_tickangle=-90, coloraxis_showscale=False, height=700)

print("""
GRÁFICO 5: Retención por Géneros (Barras)
- Qué es: Medición estática de retención de todos los géneros disponibles.
- Qué se ve: Horas de juego medianas excluyendo títulos con 0 horas.
- Elementos: Eje X (Género), Eje Y (Horas), Etiquetas numéricas rotadas.
""")

fig5.show()


GRÁFICO 5: Retención por Géneros (Barras)
- Qué es: Medición estática de retención de todos los géneros disponibles.
- Qué se ve: Horas de juego medianas excluyendo títulos con 0 horas.
- Elementos: Eje X (Género), Eje Y (Horas), Etiquetas numéricas rotadas.



In [112]:
# Mapeo de nombres para legibilidad de plataformas
plat_df = df.groupby('platform_count', as_index=False)['owners_numeric'].mean()
plat_map = {1: 'Solo Windows', 2: 'Dos Plataformas', 3: 'Multiplataforma (Win/Mac/Lin)'}
plat_df['platform_label'] = plat_df['platform_count'].map(plat_map)

# Gráfico 6: Usuarios por etiquetas de plataforma
fig6 = px.bar(plat_df, x='platform_label', y='owners_numeric', color='owners_numeric',
              color_continuous_scale='Turbo', template="plotly_dark", text_auto='.2s',
              title="Usuarios Promedio por Tipo de Soporte",
              labels={'platform_label': 'Configuración de Sistema', 'owners_numeric': 'Usuarios Promedio'})
fig6.update_layout(coloraxis_showscale=False, title_font_size=18)

# Gráfico 7: Soporte en inglés
lang_df = df.groupby('english_support', as_index=False)['owners_numeric'].mean()
fig7 = px.bar(lang_df, x='english_support', y='owners_numeric', color='english_support',
              template="plotly_dark", text_auto='.2s',
              color_discrete_sequence=['#39FF14', '#FF073A'],
              title="Usuarios Promedio por Soporte en Inglés",
              labels={'english_support': 'Soporte Inglés', 'owners_numeric': 'Usuarios Promedio'})
fig7.update_layout(showlegend=False, title_font_size=18)

print("""
GRÁFICOS 6 Y 7: Accesibilidad (Barras con Datos)
- Qué es: Medición estática del impacto de localización y ports.
- Qué se ve: Diferencia de adopción media al soportar múltiples OS o idioma inglés.
- Elementos: Eje X (Atributo), Eje Y (Promedio de usuarios).
""")

fig6.show()
fig7.show()


GRÁFICOS 6 Y 7: Accesibilidad (Barras con Datos)
- Qué es: Medición estática del impacto de localización y ports.
- Qué se ve: Diferencia de adopción media al soportar múltiples OS o idioma inglés.
- Elementos: Eje X (Atributo), Eje Y (Promedio de usuarios).



In [113]:
seasonality_static = df.groupby('quarter', as_index=False).agg(
    releases=('appid', 'count')
)
seasonality_static['quarter_label'] = 'Q' + seasonality_static['quarter'].astype(str)

fig8 = px.bar(seasonality_static, x='quarter_label', y='releases', color='quarter_label',
              template="plotly_dark", text_auto=True,
              color_discrete_sequence=px.colors.sequential.Plasma,
              title="Concentración Histórica de Lanzamientos por Cuatrimestre",
              labels={'quarter_label': 'Cuatrimestre', 'releases': 'Lanzamientos Totales'})
fig8.update_layout(title_font_size=20, showlegend=False)

print("""
GRÁFICO 8: Estacionalidad de Lanzamientos (Barras Consolidadas)
- Qué es: Sumatoria histórica estática de publicaciones.
- Qué se ve: Concentración de lanzamientos por trimestre.
- Elementos: Eje X (Q1-Q4), Eje Y (Total de juegos).
""")
fig8.show()



GRÁFICO 8: Estacionalidad de Lanzamientos (Barras Consolidadas)
- Qué es: Sumatoria histórica estática de publicaciones.
- Qué se ve: Concentración de lanzamientos por trimestre.
- Elementos: Eje X (Q1-Q4), Eje Y (Total de juegos).



In [114]:
fig10 = px.line(
    seasonal_trend.sort_values(['year', 'month_num']),
    x='month_name',
    y='owners_numeric',
    color='year',
    line_group='year',
    animation_frame='year',
    animation_group='year',
    markers=True,
    title='Ventas Mensuales por Año: Evolución Animada',
    labels={'month_name': 'Mes', 'owners_numeric': 'Ventas Estimadas (Owners)', 'year': 'Año'},
    template='plotly_dark'
)

fig10.update_xaxes(
    categoryorder='array',
    categoryarray=['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
)
fig10.update_layout(
    title_font_size=20,
    hovermode='x unified',
    yaxis=dict(range=[0, seasonal_trend['owners_numeric'].max() * 1.05])
)
print("""
GRÁFICO 10: Evolución de Ventas Mensuales por Año(Líneas Animadas)
Que es: Una serie temporal animada que narra la evolución secuencial de las ventas mensuales.
Que se ve: La progresión del éxito comercial y los patrones de consumo estacionales año tras año.
Elementos: Eje cronológico, escala de volumen, controles de reproducción y marcadores dinámicos.
""")

fig10.show()


GRÁFICO 10: Evolución de Ventas Mensuales por Año(Líneas Animadas)
Que es: Una serie temporal animada que narra la evolución secuencial de las ventas mensuales.
Que se ve: La progresión del éxito comercial y los patrones de consumo estacionales año tras año.
Elementos: Eje cronológico, escala de volumen, controles de reproducción y marcadores dinámicos.

